# 이야기담 — v2 재학습 + v1 vs v2 성능 비교 (새 노트북)
**순서대로 실행.** 런타임: GPU(A100 권장). 크레딧이 부족하면 §1~§3(환경+데이터 준비)은 CPU 런타임에서 먼저 돌리고, 학습 직전에 GPU로 바꾸되 **런타임을 바꾸면 /content가 초기화**되니 주의.

- v1 = 기존 모델 (`deploy/rtmpose-m_drawing15_ep30.pth`, 30에폭, test AP 0.919)
- v2 = v1(ep30)에서 이어서, train+val(약 168k장)로 12에폭 재파인튜닝 (lr 3e-4)
- test 8,908장은 **v1과 동일 split(seed 42)** 으로 학습에서 완전히 제외 → 공정 비교


## 1. Drive 마운트 + GPU 확인

In [6]:
!wget --spider -S https://dl.fbaipublicfiles.com/amateur_drawings/amateur_drawings.tar 2>&1 | grep -i content-length

  Content-Length: 48208397683


In [7]:
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/yiyagidam'
os.makedirs(f'{DRIVE}/work_dirs', exist_ok=True)
CFG_DIR = '/content/mmpose/configs/body_2d_keypoint/rtmpose/coco/'
PY = 'MPLBACKEND=Agg PYTHONUNBUFFERED=1 /usr/local/miniconda/bin/conda run --no-capture-output -n mmpose_env python -u'
!nvidia-smi --query-gpu=name,memory.total --format=csv
print('CPU:', os.cpu_count())
!ls {DRIVE}/deploy

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
name, memory.total [MiB]
NVIDIA L4, 23034 MiB
CPU: 12
drawing15_full.py  drawing15.py  rtmpose-m_drawing15_ep30.pth


## 2. conda 환경 + mmpose 설치 (~10분)
마지막 줄에 `CUDA True`가 찍혀야 함 (GPU 런타임일 때). `mmpose.__version__` 에러는 무시.

In [8]:
%cd /content
!wget -q -nc https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!test -d /usr/local/miniconda || bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local/miniconda > /dev/null
!/usr/local/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!/usr/local/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!test -d /usr/local/miniconda/envs/mmpose_env || /usr/local/miniconda/bin/conda create -n mmpose_env python=3.10 -y -q

E = "/usr/local/miniconda/bin/conda run -n mmpose_env"
!{E} pip install -q torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121
!{E} pip install -q "numpy<2" "opencv-python<5"
!{E} pip install -q -U openmim
!{E} mim install -q mmengine "mmcv==2.1.0" "mmdet>=3.1.0"
!test -d mmpose || git clone -q https://github.com/open-mmlab/mmpose.git
!{E} pip install -q "numpy<2" chumpy --no-build-isolation
!{E} bash -c "cd /content/mmpose && pip install -q -e ."
!{E} pip install -q "albumentations==1.3.1" future tensorboard
!{E} pip install -q "numpy<2" "opencv-python<5" "opencv-python-headless<5"
!cd / && MPLBACKEND=Agg /usr/local/miniconda/bin/conda run -n mmpose_env python -c "import torch; from mmpose.apis import init_model; print('CUDA', torch.cuda.is_available(), '| mmpose import OK')"

/content
accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: ...working... done
Channels:
 - defaults
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /usr/local/miniconda/envs/mmpose_env

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    libnsl-2.0.0               |       h5eee18b_0          31 KB
    packaging-26.3             |  py310h06a4308_0         329 KB
    pip-26.2.2                 |     pyhc872135_0         1.1 MB
    python-3.10.22             |       hd9cb797_0        24.4 MB
    setuptools-84.0.0          |  py310h06a4308_0         968 KB
    tk-8.6.15                  |       h54e0aa7_0         3.4 MB
    wheel-0.48.0        

## 3. 데이터 다운로드 + 압축 해제 (15~20분)

In [12]:
%cd /content
!wget -q -nc https://dl.fbaipublicfiles.com/amateur_drawings/amateur_drawings_annotations.json
!mkdir -p /content/data
!wget -q -O - https://dl.fbaipublicfiles.com/amateur_drawings/amateur_drawings.tar | tar -xz -C /content/data/
!wget -q -nc https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/rtmpose-m_simcc-coco_pt-aic-coco_420e-256x192-d8dd5ca4_20230127.pth -O rtmpose-m_coco-256x192.pth
!find /content/data -type f | wc -l

/content
182132


## 4. 17→15 joint 변환 + 이미지 단위 split (**v1 split_ids.json을 그대로 불러옴**)

In [13]:
import json, random
import numpy as np

COCO_IDX = {n: i for i, n in enumerate([
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"])}
J15 = ["nose", "neck", "right_shoulder", "right_elbow", "right_wrist",
       "left_shoulder", "left_elbow", "left_wrist", "pelvis",
       "right_hip", "right_knee", "right_ankle", "left_hip", "left_knee", "left_ankle"]
SKEL_1BASED = [[a + 1, b + 1] for a, b in [(0,1),(1,2),(2,3),(3,4),(1,5),(5,6),(6,7),
                                           (1,8),(8,9),(9,10),(10,11),(8,12),(12,13),(13,14)]]

def to15(k17):
    k = np.array(k17, dtype=np.float32).reshape(17, 3)
    g = lambda n: k[COCO_IDX[n]]
    mid = lambda a, b: np.array([(g(a)[0]+g(b)[0])/2, (g(a)[1]+g(b)[1])/2, min(g(a)[2], g(b)[2])])
    return np.stack([g("nose"), mid("left_shoulder", "right_shoulder"),
                     g("right_shoulder"), g("right_elbow"), g("right_wrist"),
                     g("left_shoulder"), g("left_elbow"), g("left_wrist"),
                     mid("left_hip", "right_hip"),
                     g("right_hip"), g("right_knee"), g("right_ankle"),
                     g("left_hip"), g("left_knee"), g("left_ankle")])

raw = json.load(open('/content/amateur_drawings_annotations.json'))
anns = []
for a in raw['annotations']:
    if a.get('num_keypoints', 0) == 0:
        continue
    k = to15(a['keypoints'])
    if (k[:, 2] > 0).sum() < 8:
        continue
    anns.append(dict(id=a['id'], image_id=a['image_id'], bbox=a['bbox'], kpts=k.tolist()))
img_by_id = {im['id']: im for im in raw['images']}
print('annotations:', len(anns))

# v1 때 저장해 둔 split을 그대로 불러온다 (test 8,908장 동일 보장)
SPLIT = {k: set(v) for k, v in json.load(open(f'{DRIVE}/split_ids.json')).items()}
assert not (SPLIT['train'] & SPLIT['val']) and not (SPLIT['train'] & SPLIT['test']) and not (SPLIT['val'] & SPLIT['test'])
valid_ids = {a['image_id'] for a in anns}
for _k in ['train', 'val', 'test']:
    print(_k, '어노테이션 없는 id:', len(SPLIT[_k] - valid_ids))   # 0이어야 정상
print({k: len(v) for k, v in SPLIT.items()})

annotations: 178166
train 어노테이션 없는 id: 0
val 어노테이션 없는 id: 0
test 어노테이션 없는 id: 0
{'test': 8908, 'val': 8908, 'train': 160350, 'smoke_train': 3000, 'smoke_val': 500}


## 5. 이미지 리사이즈(긴 변 ≤1024, JPEG) + COCO json 생성

In [14]:
import os
from PIL import Image
from concurrent.futures import ProcessPoolExecutor

SRC, DST, MAX = '/content/data/', '/content/data_small/', 1024
used = sorted(set().union(*[SPLIT[s] for s in ['train', 'val', 'test']]))

def proc(iid):
    fn0 = img_by_id[iid]['file_name']
    fn = os.path.splitext(fn0)[0] + '.jpg'
    img = Image.open(SRC + fn0)
    w, h = img.size
    s = min(1.0, MAX / max(w, h))
    if not os.path.exists(DST + fn):
        os.makedirs(os.path.dirname(DST + fn), exist_ok=True)
        img = img.convert('RGB')
        if s < 1.0:
            img = img.resize((round(w * s), round(h * s)), Image.BILINEAR)
        img.save(DST + fn, quality=95)
    return iid, (fn, s, round(w * s), round(h * s))

with ProcessPoolExecutor(os.cpu_count()) as ex:
    INFO = dict(ex.map(proc, used, chunksize=256))
print('resized:', len(INFO))

def write_coco(split, dst):
    ids = SPLIT[split]
    out_anns = []
    for a in anns:
        if a['image_id'] not in ids:
            continue
        s = INFO[a['image_id']][1]
        flat, nv = [], 0
        for x, y, v in a['kpts']:
            flat += [x * s, y * s, int(v)]
            nv += int(v > 0)
        bx = [v * s for v in a['bbox']]
        out_anns.append(dict(id=a['id'], image_id=a['image_id'], category_id=1, bbox=bx,
                             area=bx[2] * bx[3], iscrowd=0, keypoints=flat, num_keypoints=nv))
    imgs = [dict(id=i, file_name=INFO[i][0], width=INFO[i][2], height=INFO[i][3]) for i in ids]
    json.dump(dict(images=imgs, annotations=out_anns,
                   categories=[dict(id=1, name='human', supercategory='drawing',
                                    keypoints=J15, skeleton=SKEL_1BASED)]), open(dst, 'w'))
    print(f'{dst}: images {len(imgs)} / anns {len(out_anns)}')

for sp in SPLIT:
    write_coco(sp, f'/content/{sp}_coco.json')

resized: 178166
/content/test_coco.json: images 8908 / anns 8908
/content/val_coco.json: images 8908 / anns 8908
/content/train_coco.json: images 160350 / anns 160350
/content/smoke_train_coco.json: images 3000 / anns 3000
/content/smoke_val_coco.json: images 500 / anns 500


## 6. dataset_info + v1 config 복원

In [15]:
%%writefile /content/mmpose/configs/_base_/datasets/drawing15.py
dataset_info = dict(
    dataset_name='drawing15',
    paper_info=dict(),
    keypoint_info={
        0: dict(name='nose', id=0, color=[255, 128, 0], type='upper', swap='', sigma=0.026),
        1: dict(name='neck', id=1, color=[255, 128, 0], type='upper', swap='', sigma=0.079),
        2: dict(name='right_shoulder', id=2, color=[255, 128, 0], type='upper', swap='left_shoulder', sigma=0.079),
        3: dict(name='right_elbow', id=3, color=[255, 128, 0], type='upper', swap='left_elbow', sigma=0.072),
        4: dict(name='right_wrist', id=4, color=[255, 128, 0], type='upper', swap='left_wrist', sigma=0.062),
        5: dict(name='left_shoulder', id=5, color=[0, 255, 0], type='upper', swap='right_shoulder', sigma=0.079),
        6: dict(name='left_elbow', id=6, color=[0, 255, 0], type='upper', swap='right_elbow', sigma=0.072),
        7: dict(name='left_wrist', id=7, color=[0, 255, 0], type='upper', swap='right_wrist', sigma=0.062),
        8: dict(name='pelvis', id=8, color=[255, 128, 0], type='lower', swap='', sigma=0.107),
        9: dict(name='right_hip', id=9, color=[255, 128, 0], type='lower', swap='left_hip', sigma=0.107),
        10: dict(name='right_knee', id=10, color=[255, 128, 0], type='lower', swap='left_knee', sigma=0.087),
        11: dict(name='right_ankle', id=11, color=[255, 128, 0], type='lower', swap='left_ankle', sigma=0.089),
        12: dict(name='left_hip', id=12, color=[0, 255, 0], type='lower', swap='right_hip', sigma=0.107),
        13: dict(name='left_knee', id=13, color=[0, 255, 0], type='lower', swap='right_knee', sigma=0.087),
        14: dict(name='left_ankle', id=14, color=[0, 255, 0], type='lower', swap='right_ankle', sigma=0.089),
    },
    skeleton_info={
        0: dict(link=('nose', 'neck'), id=0, color=[0, 255, 255]),
        1: dict(link=('neck', 'right_shoulder'), id=1, color=[255, 128, 0]),
        2: dict(link=('right_shoulder', 'right_elbow'), id=2, color=[255, 128, 0]),
        3: dict(link=('right_elbow', 'right_wrist'), id=3, color=[255, 128, 0]),
        4: dict(link=('neck', 'left_shoulder'), id=4, color=[0, 255, 0]),
        5: dict(link=('left_shoulder', 'left_elbow'), id=5, color=[0, 255, 0]),
        6: dict(link=('left_elbow', 'left_wrist'), id=6, color=[0, 255, 0]),
        7: dict(link=('neck', 'pelvis'), id=7, color=[0, 255, 255]),
        8: dict(link=('pelvis', 'right_hip'), id=8, color=[255, 128, 0]),
        9: dict(link=('right_hip', 'right_knee'), id=9, color=[255, 128, 0]),
        10: dict(link=('right_knee', 'right_ankle'), id=10, color=[255, 128, 0]),
        11: dict(link=('pelvis', 'left_hip'), id=11, color=[0, 255, 0]),
        12: dict(link=('left_hip', 'left_knee'), id=12, color=[0, 255, 0]),
        13: dict(link=('left_knee', 'left_ankle'), id=13, color=[0, 255, 0]),
    },
    joint_weights=[1.] * 15,
    sigmas=[0.026, 0.079, 0.079, 0.072, 0.062, 0.079, 0.072, 0.062,
            0.107, 0.107, 0.087, 0.089, 0.107, 0.087, 0.089])

Writing /content/mmpose/configs/_base_/datasets/drawing15.py


In [16]:
import shutil, os
D = f'{DRIVE}/deploy'
assert os.path.exists(f'{D}/rtmpose-m_drawing15_ep30.pth'), 'v1 체크포인트 없음'
shutil.copy(f'{D}/drawing15_full.py', CFG_DIR + 'drawing15_full.py')   # v1 config
print(os.listdir(CFG_DIR)[:5], '...')

['drawing15_full.py', 'rtmpose-t_8xb256-420e_aic-coco-256x192.py', 'rtmpose-m_8xb256-420e_aic-coco-384x288.py', 'rtmpose-s_8xb256-420e_aic-coco-256x192.py', 'rtmpose-s_8xb256-420e_coco-256x192.py'] ...


## 7. v2 학습 데이터 구성 (train + val 대부분, val 1,000장은 체크포인트 선택용)

In [17]:
import json, random
tr = json.load(open('/content/train_coco.json')); va = json.load(open('/content/val_coco.json'))
te = json.load(open('/content/test_coco.json'))

random.seed(0)
vids = sorted(im['id'] for im in va['images']); random.shuffle(vids)
hold = set(vids[:1000])
va_keep = {i['id'] for i in va['images']} - hold

def pick(d, ids):
    return dict(images=[i for i in d['images'] if i['id'] in ids],
                annotations=[a for a in d['annotations'] if a['image_id'] in ids], categories=d['categories'])

kp = pick(va, va_keep)
new_tr = dict(images=tr['images'] + kp['images'], annotations=tr['annotations'] + kp['annotations'], categories=tr['categories'])
new_va = pick(va, hold)
assert not ({i['id'] for i in new_tr['images']} & {i['id'] for i in te['images']}), 'test 누수!'
json.dump(new_tr, open('/content/train_v2_coco.json', 'w')); json.dump(new_va, open('/content/val_v2_coco.json', 'w'))
print('train', len(new_tr['images']), '| val', len(new_va['images']), '| test', len(te['images']))

train 168258 | val 1000 | test 8908


## 8. v2 config 생성 (v1 config에서 값만 변경)
바뀌는 것: 에폭 12 / stage2 3 / lr 3e-4 / warmup 200 / 초기 가중치=v1 ep30 / 학습 데이터. augmentation 등은 그대로.

In [18]:
import re
s = open(f'{DRIVE}/deploy/drawing15_full.py').read()
def sub(pat, rep, flags=re.M, must=True):
    global s
    s2 = re.sub(pat, rep, s, count=1, flags=flags)
    if s2 == s:
        assert not must, f'미적용: {pat}'
        print('⚠️ 건너뜀:', pat)
    s = s2

sub(r"^max_epochs = \d+", "max_epochs = 12")
sub(r"^stage2_num_epochs = \d+", "stage2_num_epochs = 3")
sub(r"^base_lr = [\d.e-]+", "base_lr = 3e-4")
# warmup: LinearLR의 end 값을 줄바꿈/띄어쓰기와 무관하게 200으로
sub(r"(type='LinearLR'.*?begin=0,\s*end=)\d+", r"\g<1>200", flags=re.S, must=False)
sub(r"^load_from = .*", f"load_from = '{DRIVE}/deploy/rtmpose-m_drawing15_ep30.pth'")
s = s.replace("train_coco.json", "train_v2_coco.json").replace("val_coco.json", "val_v2_coco.json")
sub(r"^work_dir = .*", f"work_dir = '{DRIVE}/work_dirs/drawing15_v2'")
open(CFG_DIR + 'drawing15_v2.py', 'w').write(s)
V2_CFG = CFG_DIR + 'drawing15_v2.py'
!grep -nE "^(max_epochs|stage2|base_lr|load_from|work_dir)|v2_coco|LinearLR|end=" {V2_CFG}

⚠️ 건너뜀: (type='LinearLR'.*?begin=0,\s*end=)\d+
2:backend_args = dict(backend='local')
3:base_lr = 3e-4
27:            dict(backend_args=dict(backend='local'), type='LoadImage'),
102:    dist_cfg=dict(backend='nccl'),
105:load_from = '/content/drive/MyDrive/yiyagidam/deploy/rtmpose-m_drawing15_ep30.pth'
109:max_epochs = 12
185:        begin=0, by_epoch=False, end=500, start_factor=1e-05, type='LinearLR'),
191:        end=30,
197:stage2_num_epochs = 3
209:            dict(backend_args=dict(backend='local'), type='LoadImage'),
228:        ann_file='train_v2_coco.json',
235:            dict(backend_args=dict(backend='local'), type='LoadImage'),
289:    dict(backend_args=dict(backend='local'), type='LoadImage'),
338:    dict(backend_args=dict(backend='local'), type='LoadImage'),
391:        ann_file='val_v2_coco.json',
398:            dict(backend_args=dict(backend='local'), type='LoadImage'),
412:val_evaluator = dict(ann_file='/content/val_v2_coco.json', type='CocoMetric')
414:    dict(bac

T4

In [19]:
import re
BS = 64                       # 또 OOM 나면 32로
LR = 3e-4 * BS / 256          # 배치에 비례해 lr 조정 (7.5e-5)
s = open(V2_CFG).read()
s, n = re.subn(r"batch_size=\d+", f"batch_size={BS}", s)
s2 = re.sub(r"^base_lr = [\d.e-]+", f"base_lr = {LR:.2e}", s, count=1, flags=re.M)
assert s2 != s
open(V2_CFG, 'w').write(s2)
print('batch_size 치환:', n, '곳')   # 3곳(train/val/test)이어야 정상
!grep -nE "^base_lr|batch_size" {V2_CFG}

batch_size 치환: 3 곳
3:base_lr = 7.50e-05
200:    batch_size=64,
226:    batch_size=64,
389:    batch_size=64,


## 9. v2 학습 (백그라운드, 끊기면 `--resume`로 재실행)
예상: 에폭당 ~11분 × 12 ≈ 2.3시간 (A100). 로딩이 병목이라 `data_small`이 /content(로컬 SSD)에 있어야 함.

In [33]:
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

0 %, 0 MiB


In [34]:
W = '/content/drive/MyDrive/yiyagidam/work_dirs/drawing15_v2'
!cat {W}/last_checkpoint
open(f'{W}/last_checkpoint', 'w').write(f'{W}/epoch_9.pth')

/content/drive/MyDrive/yiyagidam/work_dirs/drawing15_v2/epoch_10.pth

67

In [35]:
!nohup bash -c 'for i in 1 2 3; do env MPLBACKEND=Agg PYTHONUNBUFFERED=1 /usr/local/miniconda/bin/conda run --no-capture-output -n mmpose_env python -u /content/mmpose/tools/train.py /content/mmpose/configs/body_2d_keypoint/rtmpose/coco/drawing15_v2.py --resume && break; sleep 60; done' >> /content/drive/MyDrive/yiyagidam/train_v2.log 2>&1 &

In [43]:
!pgrep -af "tools/train.py" | grep -v pgrep | head -2
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader
!tail -3 {LOG} | cut -c1-200

16611 bash -c for i in 1 2 3; do env MPLBACKEND=Agg PYTHONUNBUFFERED=1 /usr/local/miniconda/bin/conda run --no-capture-output -n mmpose_env python -u /content/mmpose/tools/train.py /content/mmpose/configs/body_2d_keypoint/rtmpose/coco/drawing15_v2.py --resume && break; sleep 60; done
16612 /usr/local/miniconda/bin/python /usr/local/miniconda/bin/conda run --no-capture-output -n mmpose_env python -u /content/mmpose/tools/train.py /content/mmpose/configs/body_2d_keypoint/rtmpose/coco/drawing15_v2.py --resume
90 %, 5414 MiB
10/25 08:33:24 - mmengine - INFO - Epoch(train) [13][ 550/2629]  base_lr: 1.000000e-03 lr: 1.000000e-03  eta: 4:58:24  time: 0.371022  data_time: 0.113452  memory: 4351  loss: 0.427706  loss_kpt: 0.42
10/25 08:33:43 - mmengine - INFO - Epoch(train) [13][ 600/2629]  base_lr: 1.000000e-03 lr: 1.000000e-03  eta: 4:58:04  time: 0.379735  data_time: 0.121568  memory: 4351  loss: 0.426175  loss_kpt: 0.42
10/25 08:34:03 - mmengine - INFO - Epoch(train) [13][ 650/2629]  base_l

In [45]:
!ls -t {W}/*.py | head -1 | xargs grep -nE "^(max_epochs|base_lr|stage2_num_epochs)"

3:base_lr = 7.5e-05
109:max_epochs = 12
197:stage2_num_epochs = 3


In [46]:
!pkill -f "tools/train.py"
!pkill -f "for i in 1 2 3"
!pgrep -af "tools/train.py" | grep -v pgrep

In [47]:
!ls -lt {W}/best_coco_AP_*.pth

-rw------- 1 root root 112793992 Oct 25 08:29 /content/drive/MyDrive/yiyagidam/work_dirs/drawing15_v2/best_coco_AP_epoch_12.pth


## 10. v1 vs v2 비교 (같은 test 8,908장, GT bbox, 추론옵션 base / no_flip / dark)
학습이 끝나 `work_dirs/drawing15_v2/best_coco_AP_*.pth`가 생긴 뒤 실행.

In [48]:
%%writefile /content/compare_models.py
import os; os.environ['MPLBACKEND'] = 'Agg'
import json, glob, numpy as np
from mmpose.apis import init_model, inference_topdown

D = '/content/drive/MyDrive/yiyagidam'; C = '/content/mmpose/configs/body_2d_keypoint/rtmpose/coco/'
latest = lambda pat: max(glob.glob(pat), key=os.path.getmtime)
MODELS = {'v1_ep30': (C + 'drawing15_full.py', f'{D}/deploy/rtmpose-m_drawing15_ep30.pth'),
          'v2':      (C + 'drawing15_v2.py',   latest(f'{D}/work_dirs/drawing15_v2/best_coco_AP_*.pth'))}
VARIANTS = {'base': {}, 'no_flip': {'model.test_cfg.flip_test': False},
            'dark': {'model.head.decoder.use_dark': True}}
NAMES = ["nose","neck","r_sho","r_elb","r_wri","l_sho","l_elb","l_wri","pelvis","r_hip","r_knee","r_ank","l_hip","l_knee","l_ank"]
ORIG13 = [i for i in range(15) if i not in (1, 8)]

coco = json.load(open('/content/test_coco.json'))
files = {im['id']: im['file_name'] for im in coco['images']}; anns = coco['annotations']

def run(cfg, ckpt, opts):
    m = init_model(cfg, ckpt, device='cuda:0', cfg_options=opts or None)
    d = np.full((len(anns), 15), np.nan)
    for n, a in enumerate(anns):
        x, y, w, h = a['bbox']
        r = inference_topdown(m, '/content/data_small/' + files[a['image_id']],
                              bboxes=np.array([[x, y, x + w, y + h]]), bbox_format='xyxy')[0]
        p = r.pred_instances.keypoints[0]
        gt = np.array(a['keypoints'], dtype=np.float32).reshape(15, 3)
        dist = np.linalg.norm(p - gt[:, :2], axis=1) / max(w, h)
        d[n] = np.where(gt[:, 2] > 0, dist, np.nan)
    return d

res = {}
for mn, (cfg, ck) in MODELS.items():
    print(mn, ck)
    for vn, ov in VARIANTS.items():
        try:
            res[(mn, vn)] = run(cfg, ck, ov); print('  done', vn)
        except Exception as e:
            print('  skip', vn, '->', repr(e)[:200])

rows = ['model,variant,PCK@0.05_mean13,PCK@0.1_mean13,' + ','.join(f'{n}@0.05' for n in NAMES)]
for (mn, vn), d in res.items():
    vis = ~np.isnan(d)
    j05 = ((d < 0.05) & vis).sum(0) / vis.sum(0); j10 = ((d < 0.1) & vis).sum(0) / vis.sum(0)
    rows.append(f'{mn},{vn},{j05[ORIG13].mean():.4f},{j10[ORIG13].mean():.4f},' + ','.join(f'{v:.3f}' for v in j05))
out = '\n'.join(rows); print(out); open(f'{D}/compare_v1_v2.csv', 'w').write(out)

# 부트스트랩: v2 - v1 (base) PCK@0.05 mean13 차이의 95% 신뢰구간
def per_inst(d):
    d = d[:, ORIG13]; ok = ~np.isnan(d)
    return np.where(ok.sum(1) > 0, ((d < 0.05) & ok).sum(1) / np.maximum(ok.sum(1), 1), np.nan)
sa, sb = per_inst(res[('v1_ep30', 'base')]), per_inst(res[('v2', 'base')])
rng = np.random.default_rng(0); diffs = []
for _ in range(2000):
    i = rng.integers(0, len(sa), len(sa)); diffs.append(np.nanmean(sb[i]) - np.nanmean(sa[i]))
print(f'v2-v1 PCK@0.05 diff: {np.mean(diffs):+.4f}  95% CI [{np.percentile(diffs,2.5):+.4f}, {np.percentile(diffs,97.5):+.4f}]  (CI가 0을 포함하면 노이즈)')

Writing /content/compare_models.py


In [49]:
!{PY} /content/compare_models.py

v1_ep30 /content/drive/MyDrive/yiyagidam/deploy/rtmpose-m_drawing15_ep30.pth
Loads checkpoint by local backend from path: /content/drive/MyDrive/yiyagidam/deploy/rtmpose-m_drawing15_ep30.pth
  done base
Loads checkpoint by local backend from path: /content/drive/MyDrive/yiyagidam/deploy/rtmpose-m_drawing15_ep30.pth
  done no_flip
Loads checkpoint by local backend from path: /content/drive/MyDrive/yiyagidam/deploy/rtmpose-m_drawing15_ep30.pth
  done dark
v2 /content/drive/MyDrive/yiyagidam/work_dirs/drawing15_v2/best_coco_AP_epoch_12.pth
Loads checkpoint by local backend from path: /content/drive/MyDrive/yiyagidam/work_dirs/drawing15_v2/best_coco_AP_epoch_12.pth
  done base
Loads checkpoint by local backend from path: /content/drive/MyDrive/yiyagidam/work_dirs/drawing15_v2/best_coco_AP_epoch_12.pth
  done no_flip
Loads checkpoint by local backend from path: /content/drive/MyDrive/yiyagidam/work_dirs/drawing15_v2/best_coco_AP_epoch_12.pth
  done dark
model,variant,PCK@0.05_mean13,PCK@0.1

### coco AP (공식 지표) — v1 / v2

In [50]:
CK_V1 = f'{DRIVE}/deploy/rtmpose-m_drawing15_ep30.pth'
!MPLBACKEND=Agg /usr/local/miniconda/bin/conda run --no-capture-output -n mmpose_env python /content/mmpose/tools/test.py {CFG_DIR}drawing15_full.py {CK_V1} 2>&1 | grep -E "coco/|Error"
CK_V2 = !ls -t {DRIVE}/work_dirs/drawing15_v2/best_coco_AP_*.pth | head -1
!MPLBACKEND=Agg /usr/local/miniconda/bin/conda run --no-capture-output -n mmpose_env python /content/mmpose/tools/test.py {V2_CFG} {CK_V2[0]} 2>&1 | grep -E "coco/|Error"

        save_best='coco/AP',
10/25 09:04:57 - mmengine - INFO - Epoch(test) [35/35]    coco/AP: 0.919476  coco/AP .5: 0.978759  coco/AP .75: 0.945408  coco/AP (M): 0.930708  coco/AP (L): 0.919457  coco/AR: 0.934205  coco/AR .5: 0.988886  coco/AR .75: 0.953637  coco/AR (M): 0.940000  coco/AR (L): 0.934156  data_time: 0.178179  time: 0.764809
        save_best='coco/AP',
10/25 09:05:37 - mmengine - INFO - Epoch(test) [140/140]    coco/AP: 0.911616  coco/AP .5: 0.978508  coco/AP .75: 0.934303  coco/AP (M): 0.928589  coco/AP (L): 0.911563  coco/AR: 0.928379  coco/AR .5: 0.986080  coco/AR .75: 0.948586  coco/AR (M): 0.938667  coco/AR (L): 0.928292  data_time: 0.037914  time: 0.162440


In [54]:
LOG = '/content/drive/MyDrive/yiyagidam/train_v2.log'

# 1) 에폭별 val 결과 (1,000장 hold-out)
!grep "Epoch(val)" {LOG} | sed -E 's/.*Epoch\(val\) \[([0-9]+)\].*coco\/AP: ([0-9.]+).*/epoch \1  val AP \2/'

# 2) 에폭별 마지막 train loss / acc_pose
!grep "Epoch(train)" {LOG} | awk '{split($0,a,"Epoch\\(train\\) \\["); split(a[2],b,"]"); last[b[1]]=$0} END{for(e in last) print e, last[e]}' | sed -E 's/^([0-9]+) .*loss: ([0-9.]+).*acc_pose: ([0-9.]+).*/epoch \1  loss \2  acc_pose \3/' | sort -k2 -n

epoch 1  val AP 0.874647
epoch 2  val AP 0.903109
epoch 3  val AP 0.908298
epoch 4  val AP 0.912676
epoch 5  val AP 0.915824
epoch 6  val AP 0.915062
epoch 7  val AP 0.916099
epoch 8  val AP 0.917007
epoch 9  val AP 0.917193
epoch 10  val AP 0.917334
epoch 11  val AP 0.918358
epoch 10  val AP 0.916316
epoch 10  val AP 0.917329
epoch 11  val AP 0.918338
epoch 12  val AP 0.918434
grep: {LOG}: No such file or directory


In [56]:
import re
LOG = '/content/drive/MyDrive/yiyagidam/train_v2.log'
lines = open(LOG, errors='ignore').read().splitlines()

train, val = {}, {}
for l in lines:
    m = re.search(r'Epoch\(train\) \[(\d+)\]\[\s*\d+/\d+\].*?\sloss: ([\d.]+)', l)
    if m: train[int(m[1])] = float(m[2])          # 같은 에폭이면 마지막 줄이 남음
    m = re.search(r'Epoch\(val\) \[(\d+)\]\[\d+/\d+\]\s+coco/AP: ([\d.]+)', l)
    if m: val[int(m[1])] = float(m[2])            # 재시작 중복은 마지막 값으로 덮어씀

for e in range(1, 13):
    print(f'epoch {e}')
    print(f'- train loss: {train.get(e, float("nan")):.3f}')
    print(f'- val: {val.get(e, float("nan")):.4f}')
print('\n- test: 0.9116  (v1 ep30: 0.9195)')

epoch 1
- train loss: nan
- val: 0.8746
epoch 2
- train loss: nan
- val: 0.9031
epoch 3
- train loss: nan
- val: 0.9083
epoch 4
- train loss: nan
- val: 0.9127
epoch 5
- train loss: nan
- val: 0.9158
epoch 6
- train loss: nan
- val: 0.9151
epoch 7
- train loss: nan
- val: 0.9161
epoch 8
- train loss: nan
- val: 0.9170
epoch 9
- train loss: nan
- val: 0.9172
epoch 10
- train loss: 0.430
- val: 0.9173
epoch 11
- train loss: 0.425
- val: 0.9183
epoch 12
- train loss: 0.420
- val: 0.9184

- test: 0.9116  (v1 ep30: 0.9195)


## 11. (선택) 학습 끝나면 v2 모델도 deploy로 내보내기

In [53]:
%%writefile /content/export_v2.py
import torch, shutil, os, glob
DRIVE = '/content/drive/MyDrive/yiyagidam'
ck = sorted(glob.glob(f'{DRIVE}/work_dirs/drawing15_v2/best_coco_AP_*.pth'), key=os.path.getmtime)[-1]
print('using', ck)
c = torch.load(ck, map_location='cpu', weights_only=False)
torch.save({'state_dict': c['state_dict'], 'meta': c.get('meta', {})}, f'{DRIVE}/deploy/rtmpose-m_drawing15_v2.pth')
shutil.copy('/content/mmpose/configs/body_2d_keypoint/rtmpose/coco/drawing15_v2.py', f'{DRIVE}/deploy/drawing15_v2.py')
print('saved from', ck)

Writing /content/export_v2.py
